# Explorar Dados — Code49 / Claudio Alphaville Tamboré

Scraping direto do site + análise exploratória dos dados crus.

**Fonte:** claudioalphavilletambore.com.br (~25 galpões)
**Plataforma:** Code49 (HTML mode — sem API JSON)

In [2]:
import datawork
datawork.setup()    

import json
import re
import ssl
import time
import urllib.request
from pathlib import Path

import pandas as pd
from selectolax.parser import HTMLParser

from datawork.display import show_sample, show_stats
from datawork.profiling import completeness_report, value_distribution

In [3]:
# === CONFIG ===
BASE_URL = "https://www.claudioalphavilletambore.com.br"
LISTING_URL = f"{BASE_URL}/imobiliaria/galpao/imoveis/38"
RAW_OUTPUT = Path("../../data/code49_claudio_raw.json")
RATE_LIMIT = 0.5  # seconds between requests

# HTTP setup
ctx = ssl.create_default_context()
ctx.check_hostname = False
ctx.verify_mode = ssl.CERT_NONE

def fetch(url: str) -> str:
    req = urllib.request.Request(url, headers={
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
    })
    return urllib.request.urlopen(req, context=ctx).read().decode("utf-8", errors="replace")

print("Setup OK")

Setup OK


In [ ]:
# === STEP 1: Extrair links de galpões ===
# Broker 38 (Claudio) tem 25 galpões. Scan completo do site encontrou 4 extras
# em outros brokers (IDs 406, 688, 773, 935).
# Estratégia: buscar na listing do broker 38 + adicionar os extras manualmente.
from petrus.infrastructure.scraping.scrapers.code49 import _extract_property_links

listing_html = fetch(LISTING_URL)
tree = HTMLParser(listing_html)
all_links = {pid: url for url, pid in _extract_property_links(tree, BASE_URL)}

# Galpões de outros brokers (encontrados via scan de IDs 1-49)
EXTRA_GALPAO_IDS = ["406", "688", "773", "935"]
for pid in EXTRA_GALPAO_IDS:
    if pid not in all_links:
        all_links[pid] = f"{BASE_URL}/{pid}/imoveis/galpao"

links = [(url, pid) for pid, url in sorted(all_links.items(), key=lambda x: int(x[0]))]
print(f"Total: {len(links)} galpões únicos encontrados")
for i, (url, pid) in enumerate(links):
    extra = " (outro broker)" if pid in EXTRA_GALPAO_IDS else ""
    print(f"  [{i+1:2d}] ID={pid:>5s}  {url}{extra}")

In [5]:
# === STEP 2: Parsear cada detail page ===
from petrus.infrastructure.scraping.scrapers.code49 import (
    _extract_prices,
    _extract_source_id,
    _parse_description_fields,
    _parse_features_bar,
    _parse_labeled_fields,
    _resolve_url,
)

results = []

for i, (url, pid) in enumerate(links):
    try:
        html = fetch(url)
        tree = HTMLParser(html)
        data = {"url": url, "source_id": pid}

        # Title
        title = tree.css_first("h1, .property-title, .titulo")
        if title:
            data["title"] = title.text(strip=True)

        # Description + extract specs from free text
        desc = tree.css_first(".property-description, .description, .descricao")
        if desc:
            data["description"] = desc.text(strip=True)
            _parse_description_fields(desc.text(strip=True), data)

        # Features bar (vagas, salas)
        features = tree.css_first(".c49-property-features")
        if features:
            _parse_features_bar(features.text(strip=True), data)

        # Labeled fields (div.table-row)
        _parse_labeled_fields(tree, data)

        # Prices (div.c49-property-price)
        _extract_prices(tree, data)

        # Images (data-foto on carousel items)
        images = []
        seen = set()
        for item in tree.css(".carousel-item[data-foto]"):
            src = item.attributes.get("data-foto") or ""
            if src and src not in seen:
                seen.add(src)
                images.append(src)
        # Fallback: background-image in carousel indicators
        for li in tree.css("#photos-property-carousel li[style]"):
            style = li.attributes.get("style") or ""
            m = re.search(r"url\(([^)]+)\)", style)
            if m and m.group(1) not in seen:
                seen.add(m.group(1))
                images.append(m.group(1))
        data["images"] = images

        results.append(data)
        print(f"  [{i+1:2d}/{len(links)}] ID={pid} OK — {len(data)} campos, {len(images)} imgs")
        time.sleep(RATE_LIMIT)

    except Exception as e:
        print(f"  [{i+1:2d}/{len(links)}] ID={pid} ERRO: {e}")

print(f"\nTotal: {len(results)} galpões parseados com sucesso")

  [ 1/25] ID=918 OK — 19 campos, 34 imgs
  [ 2/25] ID=905 OK — 15 campos, 46 imgs
  [ 3/25] ID=897 OK — 17 campos, 60 imgs
  [ 4/25] ID=879 OK — 14 campos, 78 imgs
  [ 5/25] ID=805 OK — 14 campos, 70 imgs
  [ 6/25] ID=700 OK — 15 campos, 26 imgs
  [ 7/25] ID=679 OK — 15 campos, 8 imgs
  [ 8/25] ID=567 OK — 15 campos, 2 imgs
  [ 9/25] ID=562 OK — 11 campos, 10 imgs
  [10/25] ID=561 OK — 13 campos, 14 imgs
  [11/25] ID=560 OK — 12 campos, 26 imgs
  [12/25] ID=544 OK — 11 campos, 10 imgs
  [13/25] ID=542 OK — 13 campos, 14 imgs
  [14/25] ID=541 OK — 13 campos, 2 imgs
  [15/25] ID=540 OK — 12 campos, 22 imgs
  [16/25] ID=539 OK — 14 campos, 10 imgs
  [17/25] ID=538 OK — 12 campos, 16 imgs
  [18/25] ID=507 OK — 11 campos, 30 imgs
  [19/25] ID=464 OK — 14 campos, 18 imgs
  [20/25] ID=375 OK — 16 campos, 14 imgs
  [21/25] ID=344 OK — 14 campos, 4 imgs
  [22/25] ID=265 OK — 13 campos, 12 imgs
  [23/25] ID=231 OK — 12 campos, 30 imgs
  [24/25] ID=99 OK — 14 campos, 14 imgs
  [25/25] ID=71 OK — 

In [6]:
# === STEP 3: Salvar raw data ===
# Salva sem images (muito grande) e com images separado
raw_for_df = []
for r in results:
    row = {k: v for k, v in r.items() if k != "images"}
    row["image_count"] = len(r.get("images", []))
    row["images_json"] = json.dumps(r.get("images", []))
    raw_for_df.append(row)

df_raw = pd.DataFrame(raw_for_df)

# Salvar JSON completo (com images) para o 02_clean
RAW_OUTPUT.parent.mkdir(parents=True, exist_ok=True)
with open(RAW_OUTPUT, "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)
print(f"Raw data salvo em: {RAW_OUTPUT} ({len(results)} registros)")

print(f"\nDataFrame: {df_raw.shape[0]} rows x {df_raw.shape[1]} cols")
print(f"Colunas: {df_raw.columns.tolist()}")

Raw data salvo em: ..\..\data\code49_claudio_raw.json (25 registros)

DataFrame: 25 rows x 21 cols
Colunas: ['url', 'source_id', 'title', 'description', 'ceilingHeight', 'docks', 'electricPower', 'parkingSpots', 'rooms', 'transaction_type', 'purpose', 'property_type', 'city', 'region', 'neighborhood', 'totalArea', 'builtArea', 'salePrice', 'image_count', 'images_json', 'rentPrice']


## Análise de Completude

Quais campos foram extraídos e qual a cobertura?

In [7]:
# Completude: quais campos têm dados?
completeness_report(df_raw)

,column,non_null,non_empty,pct_filled
0,url,25,25,100.0
1,source_id,25,25,100.0
2,purpose,25,25,100.0
3,property_type,25,25,100.0
4,city,25,25,100.0
5,image_count,25,25,100.0
6,images_json,25,25,100.0
7,neighborhood,25,25,100.0
8,transaction_type,25,25,100.0
9,title,25,24,96.0


In [8]:
# Amostra dos dados
show_sample(df_raw, n=5, title="Dados Crus — Code49 Claudio")


  Dados Crus — Code49 Claudio
Shape: 25 rows x 21 cols


,url,source_id,title,description,ceilingHeight,docks,electricPower,parkingSpots,rooms,transaction_type,purpose,property_type,city,region,neighborhood,totalArea,builtArea,salePrice,image_count,images_json,rentPrice
0,https://www.claudioalphavilletambore.com.br/91...,918,GALPÃO COM RENDA A VENDA EM SOROCABA CONDOMÍNI...,GALPÃO COM RENDA A VENDA EM SOROCABA CONDOMÍNI...,8,1,300,6,2,Venda,Comercial,Galpão,São Paulo - SP,Sorocaba,Brás,"1.458,00","1.317,73","4.280.000,00",34,"[""https://www.claudioalphavilletambore.com.br/...",NaN
1,https://www.claudioalphavilletambore.com.br/90...,905,GALPÃO NOVO EM ALPHAVILLE PARA LOCAÇÃO ELEVADO...,GALPÃO COMERCIAL PARA LOCAÇÃO EM ALPHAVILLE PR...,NaN,NaN,NaN,8,3,Locação,"Comercial, Industrial",Galpão,Barueri - SP,Alphaville,Alphaville Industrial,"827,23","827,23",NaN,46,"[""https://www.claudioalphavilletambore.com.br/...",NaN
2,https://www.claudioalphavilletambore.com.br/89...,897,GALPÃO LOCAÇÃO ALPHAVILLE ALAMEDA JURA PÉ DIRE...,"GALPÃO LOCAÇÃO PROXIMO AS SAÍDAS, LOCALIZADO N...",7,NaN,NaN,6,5,Locação,"Comercial, Industrial",Galpão,Barueri - SP,Alphaville,Alphaville Empresarial,654,654,NaN,60,"[""https://www.claudioalphavilletambore.com.br/...","16.000,00"
3,https://www.claudioalphavilletambore.com.br/87...,879,Galpão Para Venda Ou Locação Em Alphaville Ala...,Galpão/Depósito/Armazém e 4 banheiros para Alu...,NaN,NaN,NaN,10,3,"Venda, Locação",Comercial,Galpão,Barueri - SP,NaN,Alphaville Industrial,600,500,NaN,78,"[""https://www.claudioalphavilletambore.com.br/...",NaN
4,https://www.claudioalphavilletambore.com.br/80...,805,Galpão para Locação em Alphaville Tamboré Com ...,Galpão Prédio Comercial no Polo Empresarial Ta...,NaN,NaN,NaN,114,6,Locação,"Comercial, Industrial",Galpão,Santana de Parnaíba - SP,Alphaville,Alphaville,5315,NaN,NaN,70,"[""https://www.claudioalphavilletambore.com.br/...",NaN


In [9]:
# Distribuicao por campo chave
for col in ["city", "property_type", "transaction_type", "purpose"]:
    if col in df_raw.columns:
        print(f"\n--- {col} ---")
        display(value_distribution(df_raw, col))


--- city ---


,value,count,pct
0,Alphaville - SP,18,72.0
1,Barueri - SP,5,20.0
2,São Paulo - SP,1,4.0
3,Santana de Parnaíba - SP,1,4.0



--- property_type ---


,value,count,pct
0,Galpão,25,100.0



--- transaction_type ---


,value,count,pct
0,Locação,15,60.0
1,Venda,7,28.0
2,"Venda, Locação",3,12.0



--- purpose ---


,value,count,pct
0,"Comercial, Industrial",10,40.0
1,Industrial,8,32.0
2,Comercial,4,16.0
3,Residencial,3,12.0


## Anomalias e Outliers

Detectar preços absurdos, áreas impossíveis, campos suspeitos.

In [10]:
# Detectar anomalias nos precos
from petrus.infrastructure.mdm.transforms.numbers import parse_br_number

def safe_parse(val):
    if pd.isna(val) or not val:
        return None
    try:
        return parse_br_number(str(val))
    except (ValueError, TypeError):
        return None

anomalias = []

for _, row in df_raw.iterrows():
    sid = row.get("source_id", "?")
    
    # Preco absurdo (> R$ 100M para venda ou > R$ 1M para locacao)
    sale = safe_parse(row.get("salePrice"))
    rent = safe_parse(row.get("rentPrice"))
    if sale and sale > 100_000_000:
        anomalias.append(f"ID {sid}: preco venda absurdo R$ {sale:,.2f} (placeholder?)")
    if rent and rent > 1_000_000:
        anomalias.append(f"ID {sid}: preco locacao absurdo R$ {rent:,.2f} (placeholder?)")
    
    # Area muito grande (> 50.000 m2)
    area = safe_parse(row.get("totalArea"))
    if area and area > 50_000:
        anomalias.append(f"ID {sid}: area total {area:,.0f} m2 (verificar)")
    
    # Sem preco nenhum
    if not sale and not rent:
        anomalias.append(f"ID {sid}: sem preco (sob consulta?)")

print(f"Anomalias encontradas: {len(anomalias)}")
for a in anomalias:
    print(f"  - {a}")

Anomalias encontradas: 6
  - ID 905: sem preco (sob consulta?)
  - ID 879: sem preco (sob consulta?)
  - ID 805: sem preco (sob consulta?)
  - ID 507: sem preco (sob consulta?)
  - ID 265: sem preco (sob consulta?)
  - ID 231: preco locacao absurdo R$ 9,999,999,827,968.00 (placeholder?)


## Resumo

Raw data salvo em `data/code49_claudio_raw.json`. Proximo passo: `02_clean.ipynb` para transformar em CanonicalRecord e validar contra Silver schema.